# Vježba 3a

Proučite dataset [Multiple Linear Regression Dataset](https://www.kaggle.com/datasets/hussainnasirkhan/multiple-linear-regression-dataset) i definirajte što je target varijabla. Iz dataseta izbacite sve značajke koje nisu numeričke osim jedne, a ako negdje nedostaje vrijednost umetnite medijan. Ako nema niti jedne kategorijske značajke, onda jednu numerički pretvorite u kategorijsku.

Tu jednu kategorijsku značajku konvertirajte u numeričke korištenjem OneHotEncoder-a ili slične metode. Model trenirajte sa linearnom regresijom, lasso regresijom i ridge regresijom. Dataset pripremite sa train_test_split funkcijom i pri tom koristite random_state=21, a 20% neka vam ostane za testiranje. Izračunajte RMSE sa testnim podacima za sva tri modela. Ispišite izračunato.

Ako treniranje traje predugo, ostavite samo prvih 10% redova dataseta i probajte ponovo. To možete napraviti u samom početku koristeći poznatu funkciju train_test_split, a ovo vrijedi za sve dalje zadatke. Nakon ovog bloka dalje koristiti samo df_10.

In [52]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error

In [53]:
df = pd.read_csv("multiple_linear_regression_dataset.csv", low_memory=False)
df

,age,experience,income
0,25,1,30450
1,30,3,35670
2,47,2,31580
3,32,5,40130
4,43,10,47830
5,51,7,41630
6,28,5,41340
7,33,4,37650
8,37,5,40250
9,39,8,45150


Target varijabla je "income". U datasetu nema kategorijskih varijabli pa ćemo godine pretvoriti u kategorijsku varijablu u obliku iskustva.

In [54]:
iskustvo = {
    "student": (0, 1),
    "junior": (1, 3),
    "mid-level": (3, 7),
    "senior": (7, 11),
    "principal": (11, 25)
}
def izracunaj_iskustvo(godine):
    for kategorija, (min_godine, max_godine) in iskustvo.items():
        if min_godine <= godine < max_godine:
            return kategorija
    return None

In [55]:
df["experience"] = df["experience"].apply(izracunaj_iskustvo)
df

,age,experience,income
0,25,junior,30450
1,30,mid-level,35670
2,47,junior,31580
3,32,mid-level,40130
4,43,senior,47830
5,51,senior,41630
6,28,mid-level,41340
7,33,mid-level,37650
8,37,mid-level,40250
9,39,senior,45150


In [56]:
category_columns = ["experience"]
ohe = OneHotEncoder(sparse_output=False)
ohe_array = ohe.fit_transform(df[category_columns])
ohe_df = pd.DataFrame(ohe_array, columns=ohe.get_feature_names_out(category_columns))
df = pd.concat([df, ohe_df], axis=1)
df = df.drop(columns=category_columns)
df

,age,income,experience_junior,experience_mid-level,experience_principal,experience_senior
0,25,30450,1.0,0.0,0.0,0.0
1,30,35670,0.0,1.0,0.0,0.0
2,47,31580,1.0,0.0,0.0,0.0
3,32,40130,0.0,1.0,0.0,0.0
4,43,47830,0.0,0.0,0.0,1.0
5,51,41630,0.0,0.0,0.0,1.0
6,28,41340,0.0,1.0,0.0,0.0
7,33,37650,0.0,1.0,0.0,0.0
8,37,40250,0.0,1.0,0.0,0.0
9,39,45150,0.0,0.0,0.0,1.0


In [57]:
y = df["income"]
X = df.drop(columns=["income"])

algorithms = {
    "Linear": LinearRegression(),
    "Lasso": Lasso(alpha=0.1, random_state=21, max_iter=10000),
    "Ridge": Ridge(alpha=1.0, random_state=21, max_iter=10000)
}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21)

In [58]:
models = {
    "Linear": algorithms["Linear"].fit(X_train, y_train),
    "Lasso": algorithms["Lasso"].fit(X_train, y_train),
    "Ridge": algorithms["Ridge"].fit(X_train, y_train)
}

predictions = {
    "Linear": models["Linear"].predict(X_test),
    "Lasso": models["Lasso"].predict(X_test),
    "Ridge": models["Ridge"].predict(X_test)
}

rmse = {
    "Linear": np.sqrt(mean_squared_error(y_test, predictions["Linear"])),
    "Lasso": np.sqrt(mean_squared_error(y_test, predictions["Lasso"])),
    "Ridge": np.sqrt(mean_squared_error(y_test, predictions["Ridge"]))
}

In [59]:
print(rmse)

{'Linear': np.float64(3408.815059661026), 'Lasso': np.float64(3409.3883288556444), 'Ridge': np.float64(5250.386303141068)}
